# 🖥️ Executive Dashboarding

This notebook implements the final dimension of our analytical framework: **Executive Dashboarding**.

**Objective:** Consolidating insights into a single view for high-level decision makers.

## 1. Setup and Data Loading

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Set Style
sns.set_theme(style="whitegrid")
plt.rcParams['figure.facecolor'] = '#f4f4f4'

# Load data
df = pd.read_csv('../Amazon.csv')
df['OrderDate'] = pd.to_datetime(df['OrderDate'])
success_orders = df[df['OrderStatus'].isin(['Delivered', 'Shipped'])].copy()

## 2. The Executive KPI Snapshot

In [ ]:
metrics = {
    'Total Sales Revenue': f"${success_orders['TotalAmount'].sum():,.2f}",
    'Total Orders': f"{len(df):,}",
    'Fulfillment Rate': f"{(len(success_orders)/len(df))*100:.1f}%",
    'Average Order Value': f"${success_orders['TotalAmount'].mean():.2f}",
    'Unique Customers': f"{success_orders['CustomerID'].nunique():,}"
}

print("========== EXECUTIVE SUMMARY ==========")
for k, v in metrics.items():
    print(f"{k:<25}: {v}")
print("========================================")

## 3. Top Performer Dash (4 Quadrants)

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(20, 15))

# Q1: Top Categories
cat_rev = success_orders.groupby('Category')['TotalAmount'].sum().sort_values()
cat_rev.plot(kind='barh', ax=axes[0,0], color='#4169E1')
axes[0,0].set_title('Revenue by Category', fontsize=14, fontweight='bold')

# Q2: Sales Trend
monthly_rev = success_orders.resample('M', on='OrderDate')['TotalAmount'].sum()
monthly_rev.plot(ax=axes[0,1], marker='o', color='#2E8B57')
axes[0,1].set_title('Monthly Revenue Performance', fontsize=14, fontweight='bold')

# Q3: Payment Method Split
pm_rev = success_orders['PaymentMethod'].value_counts()
axes[1,0].pie(pm_rev, labels=pm_rev.index, autopct='%1.1f%%', startangle=140, colors=sns.color_palette('pastel'))
axes[1,0].set_title('Transaction Volume by Payment Method', fontsize=14, fontweight='bold')

# Q4: Shipping Burden by Category
ship_burden = success_orders.groupby('Category')['ShippingCost'].mean().sort_values()
ship_burden.plot(kind='barh', ax=axes[1,1], color='#CD5C5C')
axes[1,1].set_title('Average Shipping Burden by Category', fontsize=14, fontweight='bold')

plt.tight_layout()
plt.show()

## 4. Final Recommendations
1. **Inventory Focus**: Invest more in the top-performing categories identified in Chart 1.
2. **Logistics Optimization**: Investigate the shipping costs in the high-burden categories from Chart 4.
3. **Customer Retention**: Leverage the RFM segments created in high-value segments to increase seasonal revenue trends shown in Chart 2.